# Index Wanderlust PDFs into MongoDB Atlas

Run this notebook in the college Jupyter environment with its assigned CUDA GPU. It processes the PDFs only when you run it, then stores text chunks and embeddings in `wanderlust.rag_documents`. Normal chatbot requests do not run this notebook or reprocess the PDFs.

Before running, make the repository files and all 11 PDFs available in this Jupyter workspace. Never save an Atlas URI, password, token, or API key in a cell or notebook output. Use only a college-approved Jupyter environment.

In [7]:
from pathlib import Path
import sys

rag_service_root = None
project_root = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "rag_service" / "app" / "ingestion.py").is_file():
        project_root = candidate
        rag_service_root = candidate / "rag_service"
        break
    if candidate.name == "rag_service" and (candidate / "app" / "ingestion.py").is_file():
        rag_service_root = candidate
        project_root = candidate.parent
        break

if rag_service_root is None or project_root is None:
    raise FileNotFoundError(
        "Could not find rag_service/app/ingestion.py. Open this notebook from the project checkout or upload the project files."
    )

sys.path.insert(0, str(rag_service_root))
documents_dir = project_root / "data/pdf"
pdf_files = sorted(
    (path for path in documents_dir.iterdir() if path.is_file() and path.suffix.lower() == ".pdf"),
    key=lambda path: path.name.lower(),
) if documents_dir.is_dir() else []

print(f"Project folder: {project_root}")
print(f"PDF folder exists: {documents_dir.is_dir()}")
print(f"PDF files found: {len(pdf_files)}")
for pdf_file in pdf_files:
    print(f" - {pdf_file.name}")

if len(pdf_files) != 11:
    raise ValueError(f"Expected 11 PDFs; found {len(pdf_files)}. Check the uploaded project files before continuing.")

Project folder: /home/jovyan/vishnu/raj
PDF folder exists: True
PDF files found: 11
 - 01. project_overview.pdf
 - 02. user_manual.pdf
 - 03. host_guide.pdf
 - 04. room_share_and_companion_guide.pdf
 - 05. ai_travel_planner_guide.pdf
 - 06. terms_and_conditions.pdf
 - 07. billing_and_refund_policy.pdf
 - 08. security_and_privacy.pdf
 - 09. faq.pdf
 - 10. troubleshooting_and_support.pdf
 - 11. admin_guide.pdf


## Prepare the kernel

The college kernel should already have CUDA-enabled PyTorch. If an import below is missing, install only missing packages in a separate cell with `%pip install pypdf pymongo sentence-transformers python-dotenv`. Do not reinstall or replace PyTorch unless college instructions require it; then restart the kernel and verify CUDA again.

In [8]:
import torch
from sentence_transformers import SentenceTransformer

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available in this notebook kernel. Select the college GPU kernel before indexing.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch CUDA: {torch.version.cuda}")

GPU: NVIDIA A100-SXM4-80GB MIG 2g.20gb
PyTorch CUDA: 12.6


## Connect to Atlas securely

Use a least-privilege database user and ensure the college server is permitted by your Atlas Network Access settings and college policy. The URI is requested without echo and is not written into the notebook source. Never print it, save it in notebook outputs, or share it in chat.

In [12]:
%pip install pymongo
from getpass import getpass
from app.mongo import connect_to_collection

atlas_uri = getpass("MongoDB Atlas connection string: ")
mongo_client, rag_collection = connect_to_collection(
    atlas_uri,
    database_name="wanderlust",
    collection_name="rag_documents",
)
del atlas_uri
print(f"Connected to {rag_collection.full_name}")
print(f"Current chunk count: {rag_collection.count_documents({})}")

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 2.0 MB/s  0:00:0036m-:--:--
Note: you may need to restart the kernel to use updated packages.


MongoDB Atlas connection string:  ········


Connected to wanderlust.rag_documents
Current chunk count: 0


## Load and verify the embedding model

The model must match the Atlas index: `sentence-transformers/all-MiniLM-L6-v2`, 384 dimensions. The first run may download model files to the college environment.

In [13]:
from app.config import EMBEDDING_MODEL

if EMBEDDING_MODEL != "sentence-transformers/all-MiniLM-L6-v2":
    raise ValueError(f"Unexpected embedding model: {EMBEDDING_MODEL}")

embedding_model = SentenceTransformer(EMBEDDING_MODEL, device="cuda")
test_vector = embedding_model.encode(
    "Wanderlust room sharing",
    normalize_embeddings=True,
)
if len(test_vector) != 384:
    raise ValueError(f"Expected 384 dimensions, got {len(test_vector)}")

print(f"Model: {EMBEDDING_MODEL}")
print(f"Device: {embedding_model.device}")
print(f"Test vector dimensions: {len(test_vector)}")

Model: sentence-transformers/all-MiniLM-L6-v2
Device: cuda:0
Test vector dimensions: 384


## Index the PDFs

This cell processes each PDF page by page, creates overlapping chunks, embeds chunks in GPU batches, and upserts records into Atlas. Re-running it for unchanged PDFs does not create duplicates. It does not erase the collection.

In [14]:
from app.ingestion import ingest_pdf_directory

ingestion_summary = ingest_pdf_directory(
    documents_dir=documents_dir,
    collection=rag_collection,
    model=embedding_model,
    embedding_model=EMBEDDING_MODEL,
    chunk_size=900,
    overlap=180,
    batch_size=32,
)

for result in ingestion_summary:
    print(
        f"{result['sourceFile']}: "
        f"{result['chunksUpserted']} chunks upserted, "
        f"{result['staleChunksRemoved']} stale chunks removed"
    )

print(f"Total stored chunks: {rag_collection.count_documents({})}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

01. project_overview.pdf: 13 chunks upserted, 0 stale chunks removed
02. user_manual.pdf: 7 chunks upserted, 0 stale chunks removed
03. host_guide.pdf: 5 chunks upserted, 0 stale chunks removed
04. room_share_and_companion_guide.pdf: 6 chunks upserted, 0 stale chunks removed
05. ai_travel_planner_guide.pdf: 4 chunks upserted, 0 stale chunks removed
06. terms_and_conditions.pdf: 6 chunks upserted, 0 stale chunks removed
07. billing_and_refund_policy.pdf: 4 chunks upserted, 0 stale chunks removed
08. security_and_privacy.pdf: 4 chunks upserted, 0 stale chunks removed
09. faq.pdf: 18 chunks upserted, 0 stale chunks removed
10. troubleshooting_and_support.pdf: 4 chunks upserted, 0 stale chunks removed
11. admin_guide.pdf: 4 chunks upserted, 0 stale chunks removed
Total stored chunks: 75


## Verify Vector Search

This test embeds one question, searches the existing `rag_vector_index`, and prints short source previews. A search-index error usually means Atlas has not finished indexing the new vectors yet; wait for its status to be ready and retry.

In [15]:
from app.embeddings import get_embedding_model

if "embedding_model" not in globals():
    embedding_model = get_embedding_model()

question = "How does room sharing work?"
question_vector = embedding_model.encode(
    question,
    normalize_embeddings=True,
).tolist()

pipeline = [
    {
        "$vectorSearch": {
            "index": "rag_vector_index",
            "path": "embedding",
            "queryVector": question_vector,
            "numCandidates": 50,
            "limit": 5,
        }
    },
    {
        "$project": {
            "_id": 0,
            "sourceFile": 1,
            "pageNumber": 1,
            "text": 1,
            "score": {"$meta": "vectorSearchScore"},
        }
    },
]

matches = list(rag_collection.aggregate(pipeline))
if not matches:
    raise RuntimeError("Vector Search returned no chunks. Check Atlas index status and stored data.")

for match in matches:
    preview = " ".join(match["text"].split())[:240]
    print(
        f"{match['sourceFile']} p.{match['pageNumber']} "
        f"score={match['score']:.3f}: {preview}"
    )

mongo_client.close()
print("Atlas connection closed. Ingestion is complete.")

01. project_overview.pdf p.1 score=0.672: WANDERLUST DOCUMENTATION Page 1 WANDERLUST PLATFORM DOCUMENTATION DOCUMENT 01 • WL-DOC-01 Project Overview & Platform Details Comprehensive documentation of Wanderlust: platform purpose, industry context, core features, accommodation catego
Atlas connection closed. Ingestion is complete.
